# Project 13 — Subpopulations (2-component Gaussian mixture)

**Scenario.** A biophysical readout (single-molecule FRET efficiency, or a SAXS order parameter) reflects **two conformational states**. Each molecule is in a 'low' or 'high' state when measured, and the two states emit Gaussian signals with different means. We observe only the pooled, *unlabelled* signal — a bimodal histogram — and must infer the per-state means, the shared spread, and the mixing weights.

**New skill.** Latent component membership: every observation secretly belongs to a component. **Key pitfall.** *Label switching* — the mixture likelihood is invariant to permuting the component labels, so chains can swap which label is 'low' vs 'high', corrupting the per-component marginals. We fix it with an **ordered transform** on the means.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

We assume: (a) exactly two states (we *fix* K=2 — choosing K is itself a modelling decision), (b) Gaussian emissions with a **shared** spread sigma, (c) independent draws (no temporal correlation — that is Project 14's HMM). We synthesize from known truth so we can check recovery: weights (0.35, 0.65), means (-2.0, 1.5), sigma 0.7, hence separation 3.5.

In [ ]:
from data.generate_data import generate
data = generate()
y = data['y']
t = data['truth']
print(f"N={len(y)}; true means=({t['mu[0]']:.2f},{t['mu[1]']:.2f}), "
      f"sigma={t['sigma']:.2f}, w_high={t['w[1]']:.2f}, sep={t['separation']:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(y, bins=40, color='#55A868', edgecolor='white', density=True)
ax.set(xlabel='signal', ylabel='density', title='Observed signal — clearly bimodal')
plt.tight_layout()

## Step 2 — Model specification (likelihood + justified priors)

We marginalize the discrete labels analytically using `pm.NormalMixture`:

$$y_i \sim \sum_{k} w_k\,\mathcal{N}(\mu_k, \sigma), \quad w \sim \text{Dirichlet}(2,2),\; \sigma \sim \text{HalfNormal}(1).$$

**The crucial prior is on the means.** Plain `mu ~ Normal(0,3, shape=2)` leaves the labeling symmetry intact. We instead impose an **ordered transform** so $\mu_0 < \mu_1$ always. This pins label 0 to the lower state and label 1 to the higher state, breaking the symmetry that causes label switching.

In [ ]:
from model import build_model, fit, add_separation
model = build_model(data, ordered=True)
model

## Step 3 — Prior predictive checks

Datasets implied by the prior should look like *plausible bimodal signals* — varied separations and weights, but not absurd (e.g. means at ±50). We draw from the prior predictive and overlay a few simulated histograms.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=200, random_seed=RNG)
pp = prior.prior_predictive['y'].values.reshape(-1, len(y))
fig, ax = plt.subplots(figsize=(6,3.5))
for i in range(6):
    ax.hist(pp[i], bins=30, histtype='step', density=True, alpha=0.7)
ax.set(xlabel='signal', ylabel='density',
       title='Prior predictive datasets — varied but plausible')
plt.tight_layout()

## Step 4 — Inference (NUTS)

Settings: `draws=500, tune=1000, chains=4, target_accept=0.9`. Four chains are important here precisely so we can *detect* label switching via R-hat if it occurred. The ordered transform should keep things clean.

In [ ]:
idata = fit(data, draws=500, tune=1000, chains=4, seed=13)
add_separation(idata)

## Step 5 — Computational diagnostics

With the ordered transform, R-hat for `mu` should be ≈1.00 and the four chains should agree. (In the broken notebook, where ordering is removed, the chains disagree and `mu`'s R-hat blows up — the signature of label switching.)

In [ ]:
print(az.summary(idata, var_names=['w','mu','sigma','separation']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

In [ ]:
az.plot_trace(idata, var_names=['mu','w']); plt.tight_layout()

**Reading the trace.** Each `mu` component should be a tight, well-mixed band with the four chains overlapping. If instead you saw two chains parked near (-2, 1.5) and two near (1.5, -2), that bimodal-per-label trace would be label switching — fixed by ordering, not by more tuning.

## Step 6 — Posterior predictive checks

Does the fitted mixture reproduce the bimodal shape of the data? We overlay posterior-predictive densities on the observed histogram.

In [ ]:
ax = az.plot_ppc(idata, num_pp_samples=100); plt.tight_layout()

## Step 7 — Model criticism & the identifiability lesson

Even with ordering, the **raw labels** are only meaningful relative to the constraint. The robustly identifiable quantities are label-invariant: the **separation** $\mu_1-\mu_0$, the shared $\sigma$, and the weight of the higher-mean component. We compare those to truth.

In [ ]:
for name, truth in [('separation', t['separation']), ('sigma', t['sigma']),
                    ('w[1]', t['w[1]'])]:
    if name == 'w[1]':
        post = idata.posterior['w'].isel(w_dim_0=1).values.ravel()
    else:
        post = idata.posterior[name].values.ravel()
    lo, hi = np.percentile(post, [3, 97])
    print(f'{name:>11}: post mean={post.mean():.3f} 94%=[{lo:.3f},{hi:.3f}] '
          f'truth={truth:.3f}')

## Step 8 — Decision & communication

For a collaborator: report the fraction of molecules in the high state and the gap between states. E.g. 'About 65% of molecules occupy the high-FRET state; the two states differ by ~3.5 signal units (94% CI), well-resolved.' The decision (are there really two states? is the minor population real?) follows from the weight's credible interval and the posterior predictive fit.

In [ ]:
w_high = idata.posterior['w'].isel(w_dim_0=1).values.ravel()
print(f'P(high state) posterior mean = {w_high.mean():.3f}')
print(f'P(minor population > 10% of molecules) = {np.mean(np.minimum(w_high,1-w_high) > 0.1):.3f}')